# Collaborative Semantics Verbalization

This notebook implements **collaborative semantics verbalization** for GRAM, which finds the top-k most similar items for each item using a collaborative filtering model (SASRec).

**Pipeline:**
1. Train SASRec using [RecBole](https://recbole.io/) (skip if you already have a pre-trained model)
2. Load the pre-trained SASRec model and extract item embeddings
3. Compute cosine similarity between item embeddings
4. Save top-k similar items for each anchor item

**Output:** `similar_item_sasrec.txt`

**Requirements:**
```bash
pip install torch recbole numpy
```

> **Note on RecBole dataset format:**  
> RecBole requires the interaction data in `.inter` format (tab-separated).  
> Prepare `{dataset}.inter` with columns: `user_id:token\titem_id:token\ttimestamp:float`  
> and place it in `recbole_data/{dataset}/` before training.

In [ ]:
import os
import glob
import torch
import numpy as np

# ===== Dataset Configuration =====
dataset_name = "Beauty"              # Options: Beauty, Sports, Toys, Yelp
data_path = "../rec_datasets"        # Path to rec_datasets directory
recbole_data_path = "./recbole_data" # Path to RecBole-format dataset directory
model_save_dir = "./saved"           # Directory to save/load model checkpoints
os.makedirs(model_save_dir, exist_ok=True)

# ===== Retrieval Hyperparameters =====
top_k = 20    # Number of similar items to retrieve per anchor item

print(f"Dataset: {dataset_name}, top_k: {top_k}")

## Step 1: Train SASRec with RecBole

**Skip this step** if you already have a pre-trained SASRec model in `model_save_dir`.

The cell below trains SASRec using RecBole. Prepare the RecBole-format interaction data in `recbole_data/{dataset}/` before running.

In [ ]:
# Uncomment and run this cell to train SASRec.
# Skip if you already have a pre-trained model in model_save_dir.

# from recbole.quick_start import run
#
# config_dict = {
#     "data_path": recbole_data_path,
#     "checkpoint_dir": model_save_dir,
#     "USER_ID_FIELD": "user_id",
#     "ITEM_ID_FIELD": "item_id",
#     "TIME_FIELD": "timestamp",
#     "MAX_ITEM_LIST_LENGTH": 50,
#     "eval_args": {
#         "split": {"LS": "valid_and_test"},
#         "group_by": "user",
#         "order": "TO",
#         "mode": "full",
#     },
#     "train_neg_sample_args": {"uniform": 1},
#     "hidden_size": 64,
#     "num_heads": 1,
#     "num_layers": 2,
#     "hidden_dropout_prob": 0.5,
#     "attn_dropout_prob": 0.5,
#     "hidden_act": "gelu",
#     "loss_type": "CE",
#     "epochs": 300,
#     "train_batch_size": 256,
#     "eval_batch_size": 512,
#     "learning_rate": 0.001,
#     "stopping_step": 10,
#     "gpu_id": "0",
#     "seed": 2020,
# }
#
# run("SASRec", dataset_name.lower(), config_dict=config_dict)
# print(f"Training complete. Model saved to {model_save_dir}/")

## Step 2: Load Pre-trained SASRec Model

Load the saved SASRec model checkpoint and reconstruct the model using RecBole.

In [ ]:
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model

# Find the latest saved SASRec model
model_pattern = os.path.join(model_save_dir, "SASRec-*.pth")
model_files = sorted(glob.glob(model_pattern))
if not model_files:
    raise FileNotFoundError(
        f"No SASRec model found in {model_save_dir}. "
        "Please train the model first (Step 1)."
    )
model_file = model_files[-1]  # Use the latest checkpoint
print(f"Loading model: {model_file}")

# Load checkpoint and reconstruct model
checkpoint = torch.load(model_file, map_location='cpu')
config = checkpoint["config"]
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

model = get_model(config["model"])(config, train_data._dataset)
model.load_state_dict(checkpoint['state_dict'])
model.eval()
print("SASRec model loaded successfully.")

## Step 3: Compute Top-k Similar Items

Extract item embeddings from SASRec and compute cosine similarity between all item pairs. For each anchor item, we retrieve the top-k most similar items.

In [ ]:
# Extract item embeddings from SASRec
sasrec_emb = model.item_embedding.weight.data.cpu()  # shape: (num_items + 1, hidden_size)
print(f"Item embedding matrix shape: {sasrec_emb.shape}")

# Compute cosine similarity matrix
norms = torch.norm(sasrec_emb, dim=1, keepdim=True).clamp(min=1e-8)
normalized_emb = sasrec_emb / norms
cos_sim_matrix = normalized_emb @ normalized_emb.T

# Remove self-similarity (set diagonal to -inf)
cos_sim_matrix.fill_diagonal_(-1e9)

# Find top-k most similar items for each item
topk_indices = torch.topk(cos_sim_matrix, top_k, dim=1).indices.numpy()
print(f"Top-{top_k} similar items computed for {len(topk_indices)} items.")

## Step 4: Save Output

Save the similar items in the following format:
```
{anchor_ASIN} {similar_ASIN_1} {similar_ASIN_2} ... {similar_ASIN_k}
```

For example:
```
B00DJQQEGQ B00HOHLKY2 B00L0C529Q B00KKKW03U B00JCE89SU ...
```

In [ ]:
# Map RecBole item indices to original ASINs
item_tokens = dataset.field2id_token['item_id']  # index 0 is [PAD]

output_file = os.path.join(data_path, dataset_name, "similar_item_sasrec.txt")

with open(output_file, 'w') as f:
    # Skip index 0 ([PAD])
    for item_idx in range(1, len(item_tokens)):
        anchor_asin = item_tokens[item_idx]
        similar_asins = [item_tokens[idx] for idx in topk_indices[item_idx]]
        f.write(f"{anchor_asin} {' '.join(similar_asins)}\n")

print(f"Saved to: {output_file}")
print(f"Total items: {len(item_tokens) - 1}")
print("\nSample output:")
with open(output_file, 'r') as f:
    for _ in range(3):
        line = f.readline().strip()
        print(f"  {line[:100]}...")